# Multimodal Wearable Sensor Fusion for Physiological Strain Stratification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hardikjain2005/Multimodal-Wearable-Sensor-Fusion-for-Physiological-Strain-Stratification/blob/main/notebooks/run_on_colab.ipynb)
&nbsp;[GitHub repository](https://github.com/hardikjain2005/Multimodal-Wearable-Sensor-Fusion-for-Physiological-Strain-Stratification)

This notebook runs the project in Google Colab. It offers two paths:

| Path | What it does | Time |
|---|---|---|
| **A. Explore results** | Shows the committed results, tables and figures. No data download. | about 1 minute |
| **B. Reproduce everything** | Downloads the AIDLAB data (~550 MB of archives) and reruns the full pipeline from raw signals. | download time + ~15–30 min of compute on Colab's free CPU |

> Healthcare analytics project, **not a medical device**: the model estimates *activity-derived*
> strain categories (Low / Moderate / High) from wearable biosignals.

**Run the cells from top to bottom** (Runtime → Run all works too; path B is skipped unless you enable it).

## 1 · Setup

In [ ]:
# Clone the repository (skipped if it is already present)
import os, sys
if not os.path.exists("Multimodal-Wearable-Sensor-Fusion-for-Physiological-Strain-Stratification"):
    !git clone -q https://github.com/hardikjain2005/Multimodal-Wearable-Sensor-Fusion-for-Physiological-Strain-Stratification.git
%cd Multimodal-Wearable-Sensor-Fusion-for-Physiological-Strain-Stratification
print(sys.version)

In [ ]:
import sys
# Install the pinned package versions the results were produced with.
# If a pinned version has no wheel for this Colab Python, fall back to the latest compatible versions.
!{sys.executable} -m pip install -q -r requirements.txt || {sys.executable} -m pip install -q numpy pandas scipy scikit-learn matplotlib seaborn shap imbalanced-learn joblib pyEDFlib
# The pipeline runs in a fresh `python` process below, so no runtime restart is needed.
!{sys.executable} -c "import numpy, pandas, sklearn, shap; print('numpy', numpy.__version__, '| pandas', pandas.__version__, '| scikit-learn', sklearn.__version__, '| shap', shap.__version__)"

## 2 · Path A — Explore the committed results

Every number below is read from `results/metrics/`. These files are produced by `main.py`; nothing is typed in by hand.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)

cv = pd.read_csv("results/metrics/table_cv_comparison.csv")
cols = ["model", "feature_set", "accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc_ovr_macro"]
display(Markdown("### Development cross-validation (session-grouped, nested) — main experiment"))
display(cv[cv.experiment == "main"][cols].style.format(precision=3).highlight_max(subset=["f1_macro"], color="#cde2fb"))

test = pd.read_csv("results/metrics/test_results.csv")
display(Markdown("### Held-out tests (each evaluated once with the locked models)"))
display(test[["test_set"] + cols + ["sensitivity_Moderate", "sensitivity_High"]].round(3))

In [ ]:
display(Markdown("### Ablation ladder (proposed ensemble)"))
display(pd.read_csv("results/metrics/ablation_ladder.csv").round(3))
display(Markdown("### SHAP: share of mean |SHAP| per feature group"))
display(pd.read_csv("results/metrics/shap_group_share.csv"))

In [ ]:
figures = {
    "17_test_session_predictions.png": "Held-out test sessions: baseline vs proposed",
    "15_why_hr_alone_fails.png": "Why heart rate alone fails",
    "10_ablation_ladder.png": "Ablation: what each feature group adds",
    "cm_test_test_temporal.png": "Confusion matrices: later-sessions test",
    "shap_summary.png": "SHAP feature contributions",
    "01_sensor_availability.png": "Sensor availability per session",
}
for f, title in figures.items():
    display(Markdown(f"#### {title}")); display(Image(f"results/figures/{f}", width=900))

## 3 · Path B — Reproduce everything from raw data

Set `REPRODUCE = True` and run the cells below. Steps:

1. Download 18 AIDLAB sessions + AIDLAB-HAR v3 from the official source (raw data is not stored in the repo).
2. Run all pipeline stages: `data → stats → models → report → test → explain → insights`.

Results use seed 42. On the author's machine two runs were byte-identical. On a different OS or CPU, numbers may differ in the last decimal places.

In [ ]:
REPRODUCE = False   # <- set to True to download the data and rerun the full pipeline

In [ ]:
if REPRODUCE:
    !{sys.executable} scripts/download_data.py
    !du -sh data/raw
else:
    print("Skipped (REPRODUCE = False).")

In [ ]:
if REPRODUCE:
    import time; t0 = time.time()
    !{sys.executable} -u main.py --stage all 2>&1 | grep -v -i warning
    print(f"Pipeline finished in {(time.time() - t0) / 60:.1f} min")
else:
    print("Skipped (REPRODUCE = False).")

In [ ]:
if REPRODUCE:
    # Which committed metric files changed after rerunning? (empty list = identical reproduction)
    !git status --short results/metrics
    rerun = pd.read_csv("results/metrics/test_results.csv")
    display(rerun[["test_set", "model", "feature_set", "f1_macro", "accuracy"]].round(3))
else:
    print("Skipped (REPRODUCE = False).")

## 4 · Where to go next

- Run single stages, e.g. `!python main.py --stage explain`.
- Read the method and findings in the [README](https://github.com/hardikjain2005/Multimodal-Wearable-Sensor-Fusion-for-Physiological-Strain-Stratification#readme) and the project report in `report/`.
- See the dataset inspection in `docs/phase1_dataset_inspection.md` and the pre-specified analysis plan in `docs/analysis_plan.md`.